# 02 – Data Cleaning

Aufbauend auf `data/processed/reviews_imported.csv` (23'486 Zeilen, 11 Spalten) wird der
Datensatz gemäss Kapitel 3.3 in folgender Reihenfolge bereinigt:

1. Vollständige Duplikate entfernen, wobei alle Spalten ausser `Review ID` verglichen werden (Weil `Review ID` ja für jede Zeile einzigartig ist und daher niemals als Duplikat erkannt würde, wenn man sie mitzählt).
2. Zeilen ohne Review Text entfernen. Dies betrifft die 845 zuvor identifizierten fehlenden Werte in dieser Spalte.
3. Fehlende `Division/Department/Class Name` **nicht** entfernen, nur dokumentieren. Diese Zeilen bleiben erhalten, da die betroffenen Informationen für die spätere Sentiment und Regressionsanalyse nicht zwingend benötigt werden.

Ergebnis: `data/processed/reviews_cleaned.csv`, Basis für `03_VADER.ipynb`.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "reviews_imported.csv")
df.shape

(23486, 11)

## Fehlende Werte (Ausgangslage)

In [3]:
df.isna().sum()

Review ID                     0
Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64

Die Zahlen entsprechen denen aus dem Notebook `01_Data_Import.ipynb`.

## Schritt 1: Vollständige Duplikate entfernen

`Review ID` entspricht dem ursprünglichen Zeilenindex aus dem CSV Export und wird beim Duplikatvergleich ausgeschlossen. Zwei Zeilen gelten als Duplikat, wenn sie in allen übrigen Spalten übereinstimmen (`Clothing ID`, `Age`, `Title`, `Review Text`, `Rating`, `Recommended IND`, `Positive Feedback Count`, `Division`, `Department`, `Class Name`).


In [4]:
compare_cols = [c for c in df.columns if c != "Review ID"]

# Duplikate identifizieren, BEVOR sie entfernt werden
is_duplicate = df.duplicated(subset=compare_cols, keep="first")
duplicates_with_missing_text = df.loc[is_duplicate, "Review Text"].isna().sum()
print(f"Duplikate mit fehlendem Review Text: {duplicates_with_missing_text}")

n_before = len(df)
df = df.drop_duplicates(subset=compare_cols, keep="first")
n_dropped_dupes = n_before - len(df)

print(f"Vollständige Duplikate entfernt: {n_dropped_dupes}")
df.shape

Duplikate mit fehlendem Review Text: 20
Vollständige Duplikate entfernt: 21


(23465, 11)

Von den ursprünglich 23'486 Zeilen wurden 21 vollständige Duplikate entfernt. Der bereinigte Datensatz umfasst somit 23'465 Zeilen.


## Schritt 2: Zeilen ohne Review Text entfernen

Der Freitext bildet die Grundlage der Sentiment Analysis mit VADER und damit auch der anschliessenden Regression. Zeilen ohne `Review Text` können für diese Analysen nicht verwendet werden und werden daher entfernt.


In [5]:
n_before = len(df)
df = df.dropna(subset=["Review Text"])
n_dropped_no_text = n_before - len(df)

print(f"Zeilen ohne Review Text entfernt: {n_dropped_no_text}")
df.shape

Zeilen ohne Review Text entfernt: 825


(22640, 11)

Die Differenz zur ursprünglich identifizierten Zahl fehlender Review-Texte (845) ergibt sich daraus, dass ein Teil dieser Zeilen bereits durch die Duplikatentfernung in Schritt 1 entfernt wurde.

## Schritt 3: Fehlende Produktkategorisierung dokumentieren (nicht entfernen)

`Division Name`, `Department Name` und `Class Name` werden bewusst nicht entfernt. Fehlende Werte in diesen Spalten werden lediglich dokumentiert, da die betreffenden Variablen für die spätere Analyse nicht zwingend erforderlich sind.

In [6]:
category_cols = ["Division Name", "Department Name", "Class Name"]
missing_categories = df[category_cols].isna().sum()
n_rows_missing_category = df[category_cols].isna().any(axis=1).sum()

print("Fehlende Werte je Spalte:")
print(missing_categories)
print(f"\nBetroffene Zeilen insgesamt (verbleiben im Datensatz): {n_rows_missing_category}")

Fehlende Werte je Spalte:
Division Name      13
Department Name    13
Class Name         13
dtype: int64

Betroffene Zeilen insgesamt (verbleiben im Datensatz): 13


Alle drei Spalten weisen dieselbe Anzahl fehlender Werte auf (je 13), die zugleich der Anzahl betroffener Zeilen entspricht. Dies zeigt, dass in den betroffenen Zeilen jeweils alle drei Kategorien gleichzeitig fehlen und nicht nur eine oder zwei davon.

In [7]:
missing_category_mask = df[["Division Name", "Department Name", "Class Name"]].isna().any(axis=1)

In [8]:
df.loc[missing_category_mask, category_cols]

,Division Name,Department Name,Class Name
9444,NaN,NaN,NaN
13767,NaN,NaN,NaN
13768,NaN,NaN,NaN
16216,NaN,NaN,NaN
16221,NaN,NaN,NaN
16223,NaN,NaN,NaN
18626,NaN,NaN,NaN
18671,NaN,NaN,NaN
20088,NaN,NaN,NaN
21532,NaN,NaN,NaN


Diese 13 Zeilen verbleiben, wie eingangs erläutert, im Datensatz und werden lediglich für spätere Robustheitsprüfungen entsprechend markiert.

## Title

Fehlende Werte in `Title` bleiben als `NaN` bestehen, da der Titel optional ist und für die weitere Analyse nicht benötigt wird.

## Textnormalisierung

Zusätzlich werden die Wortanzahl (`Review Word Count`) und die Zeichenanzahl (`Review Char Count`) je Review berechnet, um einen Eindruck der Textlänge im Datensatz zu erhalten.


In [9]:
def normalize_whitespace(text: str) -> str:
    return " ".join(text.split())

df["Review Text"] = df["Review Text"].astype(str).map(normalize_whitespace)
df["Title"] = df["Title"].map(normalize_whitespace, na_action="ignore")

## Zusätzliche Merkmale

In [10]:
df["Review Word Count"] = df["Review Text"].str.split().str.len()
df["Review Char Count"] = df["Review Text"].str.len()

df[["Review Word Count", "Review Char Count"]].describe()

,Review Word Count,Review Char Count
count,22640.000000,22640.000000
mean,60.197482,308.322350
std,28.534986,143.714139
min,2.000000,9.000000
25%,36.000000,186.000000
50%,59.000000,301.000000
75%,88.000000,458.000000
max,115.000000,508.000000


Auf Basis von `Review Text` werden zusätzlich die Wortanzahl (`Review Word Count`) und die Zeichenanzahl (`Review Char Count`) je Review berechnet, um einen Eindruck der Textlänge im Datensatz zu erhalten.


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"


def show_txt(filename):
    """Liest eine bestehende .txt-Ergebnisdatei ein und gibt sie unverändert aus."""
    path = RESULTS_DIR / filename
    print(f"--- {filename} " + "-" * max(0, 60 - len(filename)))
    print(path.read_text())


def load_csv(filename, **kwargs):
    """Liest eine bestehende .csv-Ergebnisdatei ein (kein Neuberechnen, nur Einlesen)."""
    return pd.read_csv(RESULTS_DIR / filename, **kwargs)


## Kontrolle & Speichern

Vor dem Speichern wird der finale Datensatz auf Vollständigkeit geprüft. Fehlende Werte dürfen nur noch bei `Title` (bewusst belassen) sowie bei `Division Name`, `Department Name` und `Class Name` (dokumentiert in Schritt 3) bestehen. Alle für die Sentiment Analysis und die Regression zentralen Spalten sind vollständig gefüllt.

In [11]:
print(f"Zeilen gesamt: {len(df)}")
df.isna().sum()

Zeilen gesamt: 22640


Review ID                     0
Clothing ID                   0
Age                           0
Title                      2965
Review Text                   0
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                13
Department Name              13
Class Name                   13
Review Word Count             0
Review Char Count             0
dtype: int64

## Plausibilitätsprüfung numerischer Variablen

Abschliessend werden die Wertebereiche der zentralen numerischen Variablen `Age`, `Rating` und `Positive Feedback Count` auf Plausibilität geprüft. Alterswerte ausserhalb von 0 bis 100 Jahren, Sternebewertungen ausserhalb von 1 bis 5 sowie negative Werte bei `Positive Feedback Count` würden auf mögliche Dateneingabefehler hindeuten. Zusätzlich werden die Datentypen dieser drei Variablen kontrolliert.

In [12]:
print("Age")
print(f"  Min: {df['Age'].min()}, Max: {df['Age'].max()}")
n_age_unrealistic = ((df["Age"] <= 0) | (df["Age"] > 100)).sum()
print(f"  Unrealistische Werte (<= 0 oder > 100): {n_age_unrealistic}")

print("\nRating")
print(f"  Min: {df['Rating'].min()}, Max: {df['Rating'].max()}")
n_rating_outside = (~df["Rating"].between(1, 5)).sum()
print(f"  Werte ausserhalb von 1-5: {n_rating_outside}")

print("\nPositive Feedback Count")
print(f"  Min: {df['Positive Feedback Count'].min()}, Max: {df['Positive Feedback Count'].max()}")
n_feedback_negative = (df["Positive Feedback Count"] < 0).sum()
print(f"  Negative Werte: {n_feedback_negative}")

print("\nDatentypen")
print(df[["Age", "Rating", "Positive Feedback Count"]].dtypes)

Age
  Min: 18, Max: 99
  Unrealistische Werte (<= 0 oder > 100): 0

Rating
  Min: 1, Max: 5
  Werte ausserhalb von 1-5: 0

Positive Feedback Count
  Min: 0, Max: 122
  Negative Werte: 0

Datentypen
Age                        int64
Rating                     int64
Positive Feedback Count    int64
dtype: object


Die Wertebereiche von `Age` (18-99), `Rating` (1-5) und `Positive Feedback Count` (0-122) sind durchgehend plausibel, es liegen keine unrealistischen oder negativen Werte vor. Alle drei Variablen sind korrekt als `int64` kodiert.

Der bereinigte Datensatz wird abschliessend unter `data/processed/reviews_cleaned.csv` gespeichert und bildet die Grundlage für die Sentiment Analysis in `03_VADER.ipynb`.


In [13]:
out_path = PROCESSED_DIR / "reviews_cleaned.csv"
df.to_csv(out_path, index=False)
out_path

PosixPath('/Users/laraeibel/Desktop/Bachelorarbeit_Python/data/processed/reviews_cleaned.csv')